# Tanaos Topic Classification : Extra Meta Features

## Imports

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB,GaussianNB
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score, f1_score,roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
import gensim
import gensim.downloader as api
from gensim.models import KeyedVectors
import os
from scipy.sparse import hstack
from scipy.special import softmax
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

## Load Dataset

In this section, we load the training dataset and inspect its basic structure.
The goal is to understand:

- how many examples are available;
- which columns exist;
- how labels are stored;
- whether there are missing values;
- whether the dataset appears balanced across classes.

In [ ]:
df = pd.read_csv("data/train_data.csv")

In [ ]:
print("Shape of dataset:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
df.head()

In [ ]:
topics = ["politics","health","technology","entertainment","money_finance","relationships_dating","education_learning","work_careers","science","society_culture","gaming","lifestyle_hobbies","sports","automotive","other"]

### Label Mapping

The dataset stores class labels numerically, so we define a dictionary that maps each numeric label to its corresponding topic name.

In [ ]:
topic_dict = dict(enumerate(topics))
print(topic_dict)

In [ ]:
print("Columns:", df.columns.tolist())
print("Unique labels:", sorted(df["labels"].unique()))
print("Number of classes:", df["labels"].nunique())

### Load Embeddings

In [ ]:
model_path = "model/word2vec-google-news-300.kv"

folder_name = os.path.dirname(model_path)
if folder_name:
    os.makedirs(folder_name, exist_ok=True)

if os.path.exists(model_path):
    print("Found saved model! Loading from disk...")
    wv = KeyedVectors.load(model_path)
    print("Model loaded successfully!")

else:
    print("Model not found locally. Downloading from Gensim (this may take a few minutes)...")
    wv = api.load('word2vec-google-news-300')

    print("Download complete! Saving to disk for next time...")
    wv.save(model_path)
    print("Model saved successfully!")

## Preprocessing Strategy

Before training any classifier, the text is normalized in order to reduce noise and make the feature extraction more consistent.

The preprocessing steps used here are:

- removal of non-alphabetic characters;
- conversion to lowercase;
- tokenization by whitespace splitting;
- stopword removal;
- stemming with PorterStemmer.

A small exception is made for negation-related words, since removing them may discard useful meaning.

## Cleanup and normalization


In [ ]:
negations_to_keep = {
    "no", "nor", "not", "ain", "aren", "aren't", "couldn", "couldn't", "didn", "didn't",
    "doesn", "doesn't", "hadn", "hadn't", "hasn", "hasn't", "haven", "haven't", "isn",
    "isn't", "mightn", "mightn't", "mustn", "mustn't", "needn", "needn't", "shan",
    "shan't", "shouldn", "shouldn't", "wasn", "wasn't", "weren", "weren't", "won",
    "won't", "wouldn", "wouldn't"
}

In [ ]:
corpus = []
ps = PorterStemmer()
sw = set(stopwords.words('english'))
sw = sw-negations_to_keep
for i in range(0, df['text'].size):
    # get review and remove non alpha chars
    review = re.sub('[^a-zA-Z]', ' ', df['text'][i])
    # to lower-case
    review = review.lower()
    # split into tokens, apply stemming and remove stop words
    review = ' '.join([ps.stem(w) for w in review.split() if w not in sw])
    corpus.append(review)

print("Number of processed documents:", len(corpus))
print("\nExample original text:")
print(df['text'].iloc[0])

print("\nExample processed text:")
print(corpus[0])

In [ ]:
df["processed_text"] = corpus
df.head()

In [ ]:
unstemmed_corpus = []
for i in range(0, df['text'].size):
    review = re.sub('[^a-zA-Z]', ' ', df['text'][i])
    review = review.lower()
    review = ' '.join([w for w in review.split() if w not in sw])
    unstemmed_corpus.append(review)

df["unstemmed_text"] = unstemmed_corpus

In [ ]:
df.head()

## Pre-processing and Feature Representation

### Bag of Words

In this first baseline, we use a sparse Bag-of-Words representation with `CountVectorizer`.

This means each document is represented by a vector of token counts, where:

- each feature corresponds to a vocabulary term;
- the value indicates how many times that term appears in the document.

This is a simple and standard baseline for text classification.

In [ ]:
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(corpus)

print("Feature matrix shape:", X_bow.shape)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

In [ ]:
feature_names = vectorizer.get_feature_names_out()

print("First 100 features:")
print(feature_names[:100])

In [ ]:
y = df['labels']

print("X_bow shape:", X_bow.shape)
print("y shape:", y.shape)


In [ ]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df["processed_text"])

print("TF-IDF feature matrix shape:", X_tfidf.shape)
print("TF-IDF vocabulary size:", len(tfidf_vectorizer.get_feature_names_out()))

### 1-hot vectors

In [ ]:
vectorizer = CountVectorizer(binary=True)
X_onehot = vectorizer.fit_transform(corpus)

print("Feature matrix shape:", X_onehot.shape)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

### N-Grams

In [ ]:
vectorizer = CountVectorizer(ngram_range=(1, 2))

X_grams = vectorizer.fit_transform(corpus)

print("Feature matrix shape:", X_grams.shape)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

### Embeddings

In [ ]:
def text_to_mean_vector(embeddings, text):
    tokens = text.split()

    valid_vectors = []
    for token in tokens:
        try:
            valid_vectors.append(embeddings.get_vector(token))
        except KeyError:
            pass

    if not valid_vectors:

        return np.zeros(embeddings.vector_size)

    return np.mean(valid_vectors, axis=0)

In [ ]:
def text_to_max_vector(embeddings, text):
    tokens = text.split()

    valid_vectors = []
    for token in tokens:
        try:
            valid_vectors.append(embeddings.get_vector(token))
        except KeyError:
            pass

    if not valid_vectors:
        return np.zeros(embeddings.vector_size)

    return np.max(valid_vectors, axis=0)

In [ ]:
embeddings_mean_corpus = []
for c in unstemmed_corpus:
    embeddings_mean_corpus.append(text_to_mean_vector(wv, c))

X_embedding_mean = np.array(embeddings_mean_corpus)

print(X_embedding_mean.shape, y.shape)

In [ ]:
embeddings_max_corpus = []
for c in unstemmed_corpus:
    embeddings_max_corpus.append(text_to_max_vector(wv, c))

X_embedding_max = np.array(embeddings_max_corpus)

print(X_embedding_max.shape, y.shape)

## Split Dataset

The dataset is split into training and test sets.

A stratified split is used so that the label distribution remains approximately the same in both sets.  
This is important in multiclass classification, especially if some classes are less frequent than others.

In [ ]:
meta_features = df[['text_length_chars', 'text_length_words', 'avg_word_length']].values

In [ ]:
X_bow_combined = hstack((X_bow, meta_features))

X_train_bow, X_test_bow, y_train_bow, y_test_bow = train_test_split(
    X_bow_combined, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print(X_train_bow.shape, y_train_bow.shape)
print(X_test_bow.shape, y_test_bow.shape)

print("\nLabel distribution in the training set:")
print(y_train_bow.value_counts().sort_index())

print("\nLabel distribution in the test set:")
print(y_test_bow.value_counts().sort_index())

To ensure a fair comparison with the previous baseline, we apply the same train/test split strategy to the TF-IDF representation.


In [ ]:
X_train_tfidf, X_test_tfidf, meta_train, meta_test, y_train_tfidf, y_test_tfidf = train_test_split(
    X_tfidf, meta_features, y, test_size=0.20, random_state=42, stratify=y
)

print(X_train_tfidf.shape, y_train_tfidf.shape)
print(X_test_tfidf.shape, y_test_tfidf.shape)

In [ ]:
X_onehot_combined = hstack((X_onehot, meta_features))

X_train_onehot, X_test_onehot, y_train_onehot, y_test_onehot = train_test_split(
    X_onehot_combined, y, test_size=0.20, random_state=42, stratify=y
)

print(X_train_onehot.shape, y_train_onehot.shape)
print(X_test_onehot.shape, y_test_onehot.shape)

In [ ]:
X_grams_combined = hstack((X_grams, meta_features))

X_train_grams, X_test_grams, y_train_grams, y_test_grams = train_test_split(
    X_grams_combined, y, test_size=0.20, random_state=42, stratify=y
)

print(X_train_grams.shape, y_train_grams.shape)
print(X_test_grams.shape, y_test_grams.shape)

In [ ]:
X_emb_mean_combined = np.hstack((X_embedding_mean, meta_features))

X_train_embeddings_mean, X_test_embeddings_mean, y_train_embeddings_mean, y_test_embeddings_mean = train_test_split(
    X_emb_mean_combined, y, test_size=0.20, random_state=42, stratify=y
)

print(X_train_embeddings_mean.shape, y_train_embeddings_mean.shape)
print(X_test_embeddings_mean.shape, y_test_embeddings_mean.shape)

In [ ]:
X_emb_max_combined = np.hstack((X_embedding_max, meta_features))

X_train_embeddings_max, X_test_embeddings_max, y_train_embeddings_max, y_test_embeddings_max = train_test_split(
    X_emb_max_combined, y, test_size=0.20, random_state=42, stratify=y
)
print(X_train_embeddings_max.shape, y_train_embeddings_max.shape)
print(X_test_embeddings_max.shape, y_test_embeddings_max.shape)

In [ ]:
# BoW (MaxAbsScaler keeps zeros intact and values positive)
scaler_bow = MaxAbsScaler()
X_train_bow_final = scaler_bow.fit_transform(X_train_bow)
X_test_bow_final = scaler_bow.transform(X_test_bow)

# N-grams
scaler_grams = MaxAbsScaler()
X_train_grams_final = scaler_grams.fit_transform(X_train_grams)
X_test_grams_final = scaler_grams.transform(X_test_grams)

# One-hot
scaler_onehot = MaxAbsScaler()
X_train_onehot_final = scaler_onehot.fit_transform(X_train_onehot)
X_test_onehot_final = scaler_onehot.transform(X_test_onehot)

# TF-IDF Meta Features
scaler_meta_minmax = MinMaxScaler()
meta_train_scaled_mm = scaler_meta_minmax.fit_transform(meta_train)
meta_test_scaled_mm = scaler_meta_minmax.transform(meta_test)

X_train_tfidf_final = hstack((X_train_tfidf, meta_train_scaled_mm))
X_test_tfidf_final = hstack((X_test_tfidf, meta_test_scaled_mm))

# Mean Embeddings (StandardScaler for the win!)
scaler_emb_mean = StandardScaler()
X_train_embeddings_mean_final = scaler_emb_mean.fit_transform(X_train_embeddings_mean)
X_test_embeddings_mean_final = scaler_emb_mean.transform(X_test_embeddings_mean)

# Max Embeddings
scaler_emb_max = StandardScaler()
X_train_embeddings_max_final = scaler_emb_max.fit_transform(X_train_embeddings_max)
X_test_embeddings_max_final = scaler_emb_max.transform(X_test_embeddings_max)

## Model Training and Evaluation

In [ ]:
def evaluate_models(X_train, X_test, y_train, y_test, target_names=None, is_embeddings=False):
    """
    Trains multiple models, evaluates their performance, and returns:
    1. A metrics DataFrame (overall scores including ROC-AUC)
    2. A dictionary of confusion matrices
    3. A dictionary of classification reports
    """

    if is_embeddings:
        nb_name = 'Gaussian Naive Bayes'
        nb_model = GaussianNB()
    else:
        nb_name = 'Multinomial Naive Bayes'
        nb_model = MultinomialNB(alpha=1.0)

    models = {
        nb_name: nb_model,

        'Logistic Regression (L2)': LogisticRegression(
            max_iter=5000,
            class_weight='balanced',
            random_state=42
        ),

        'Logistic Regression (L1)': LogisticRegression(
            l1_ratio=1,
            solver='saga',
            max_iter=5000,
            tol=0.01,
            class_weight='balanced',
            random_state=42
        ),

        'Linear SVM (L2)': LinearSVC(
            max_iter=5000,
            class_weight='balanced',
            random_state=42
        ),

        'Linear SVM (L1)': LinearSVC(
            penalty='l1',
            dual=False,
            max_iter=5000,
            class_weight='balanced',
            random_state=42
        ),

        'Random Forest': RandomForestClassifier(
            n_estimators=100,
            max_depth=50,
            class_weight='balanced',
            n_jobs=-1,
            random_state=42
        ),

        'XGBoost': XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            n_jobs=-1,
            random_state=42
        )
    }

    metrics_data = []
    cm_dict = {}
    cr_dict = {}

    for model_name, model in models.items():

        # Fit the model
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)
        else:
            y_scores = model.decision_function(X_test)
            y_prob = softmax(y_scores, axis=1)

        y_prob = y_prob / y_prob.sum(axis=1, keepdims=True)

        # Calculate metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        f1_m = f1_score(y_test, y_pred, average='macro', zero_division=0)
        roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')

        metrics_data.append({
            'Model': model_name,
            'Accuracy': acc,
            'Precision (weighted)': prec,
            'Recall (weighted)': rec,
            'F1 (weighted)': f1_w,
            'F1 (macro)': f1_m,
            'ROC-AUC (weighted)': roc_auc
        })

        # Store confusion matrix
        cm_dict[model_name] = confusion_matrix(y_test, y_pred)

        # Store classification report
        cr_dict[model_name] = classification_report(
            y_test,
            y_pred,
            target_names=target_names,
            output_dict=True,
            zero_division=0
        )

    metrics_df = pd.DataFrame(metrics_data).set_index('Model')

    return metrics_df, cm_dict, cr_dict



In [ ]:
def display_model_evaluations(confusion_matrices, class_reports, class_names):
    """5000
    Iterates through model results to neatly display classification reports
    and plot confusion matrix heatmaps.

    Args:
        confusion_matrices (dict): Dictionary mapping model names to confusion matrices.
        class_reports (dict): Dictionary mapping model names to classification reports (as dicts).
        class_names (list): List of string labels for the classes (e.g., your topics).
    """
    for model_name in confusion_matrices.keys():
        print(f"\n{'='*50}")
        print(f"MODEL: {model_name}")
        print(f"{'='*50}\n")

        print("--- Classification Report ---")
        report_df = pd.DataFrame(class_reports[model_name]).transpose()
        display(report_df.round(4))
        print("\n")

        print("--- Confusion Matrix ---")
        cm = confusion_matrices[model_name]

        plt.figure(figsize=(8, 6))

        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)

        plt.ylabel('Actual Class')
        plt.xlabel('Predicted Class')
        plt.title(f'{model_name} - Confusion Matrix')
        plt.xticks(rotation=45, ha='right')

        plt.tight_layout()
        plt.show()

### BOW

As a first baseline model, we use **Multinomial Naive Bayes**.

This classifier is commonly used in text classification because it works well with count-based sparse representations such as Bag-of-Words.  
Its purpose here is to provide a simple reference point against which stronger models can later be compared.

In [ ]:
metrics_df_bow, confusion_matrices_bow, class_reports_bow = evaluate_models(
    X_train_bow_final, X_test_bow_final, y_train_bow, y_test_bow, target_names=topics
)

In [ ]:
display(metrics_df_bow.sort_values(by='F1 (macro)', ascending=False))

In [ ]:
display_model_evaluations(confusion_matrices_bow, class_reports_bow, topics)

### TF-IDF

Logistic Regression is one of the strongest traditional baselines for multiclass text classification.

It often performs better than Naive Bayes because it learns direct decision boundaries between classes instead of relying on stronger independence assumptions.

In [ ]:
metrics_df_tfidf, confusion_matrices_tfidf, class_reports_tfidf = evaluate_models(
    X_train_tfidf_final, X_test_tfidf_final, y_train_tfidf, y_test_tfidf, target_names=topics
)

In [ ]:
display(metrics_df_tfidf.sort_values(by='F1 (macro)', ascending=False))

In [ ]:
display_model_evaluations(confusion_matrices_tfidf, class_reports_tfidf, topics)

### 1-hot vectors

In [ ]:
metrics_df_onehot, confusion_matrices_onehot, class_reports_onehot = evaluate_models(
    X_train_onehot_final, X_test_onehot_final, y_train_onehot, y_test_onehot, target_names=topics
)

In [ ]:
display(metrics_df_bow.sort_values(by='F1 (macro)', ascending=False))

In [ ]:
display_model_evaluations(confusion_matrices_onehot, class_reports_onehot, topics)

### N-Grams

In [ ]:
metrics_df_grams, confusion_matrices_grams, class_reports_grams= evaluate_models(
    X_train_grams_final, X_test_grams_final, y_train_grams, y_test_grams, target_names=topics
)

In [ ]:
display(metrics_df_grams.sort_values(by='F1 (macro)', ascending=False))

In [ ]:
display_model_evaluations(confusion_matrices_grams, class_reports_grams, topics)

### Embeddings Mean

In [ ]:
metrics_df_embeddings_mean, confusion_matrices_embeddings_mean, class_reports_embeddings_mean = evaluate_models(
    X_train_embeddings_mean_final, X_test_embeddings_mean_final, y_train_embeddings_mean, y_test_embeddings_mean, target_names=topics,is_embeddings=True
)

In [ ]:
display(metrics_df_embeddings_mean.sort_values(by='F1 (macro)', ascending=False))

In [ ]:
display_model_evaluations(confusion_matrices_embeddings_mean, class_reports_embeddings_mean, topics)

### Embeddings Max

In [ ]:
metrics_df_embeddings_max, confusion_matrices_embeddings_max, class_reports_embeddings_max = evaluate_models(
    X_train_embeddings_max_final, X_test_embeddings_max_final, y_train_embeddings_max, y_test_embeddings_max, target_names=topics,is_embeddings=True
)

In [ ]:
display(metrics_df_embeddings_max.sort_values(by='F1 (macro)', ascending=False))

In [ ]:
display_model_evaluations(confusion_matrices_embeddings_max, class_reports_embeddings_max, topics)

## Model Comparison

To compare the different approaches more clearly, we summarize the main evaluation metrics in a single table.

This makes it easier to identify whether TF-IDF improves over Bag-of-Words, and whether Logistic Regression or Linear SVM outperform the Naive Bayes baseline.

In [ ]:
metrics_df_onehot.index = metrics_df_onehot.index.astype(str) + ' (One-Hot)'
metrics_df_tfidf.index = metrics_df_tfidf.index.astype(str) + ' (TF-IDF)'
metrics_df_bow.index = metrics_df_bow.index.astype(str) + ' (BoW)'
metrics_df_grams.index = metrics_df_grams.index.astype(str) + ' (grams)'
metrics_df_embeddings_mean.index = metrics_df_embeddings_mean.index.astype(str) + ' (embeddings mean)'
metrics_df_embeddings_max.index = metrics_df_embeddings_max.index.astype(str) + ' (embeddings max)'

In [ ]:
all_metrics = pd.concat([
    metrics_df_onehot,
    metrics_df_tfidf,
    metrics_df_bow,
    metrics_df_grams,
    metrics_df_embeddings_mean,
    metrics_df_embeddings_max
])

all_metrics_sorted = all_metrics.sort_values(by='F1 (weighted)', ascending=False)

display(all_metrics_sorted.head(10))

### Baseline Results & Model Selection

Looking at the updated consolidated leaderboard, several clear mathematical and structural trends immediately stand out:

**1. The Power of Linear Algorithms:** Linear Support Vector Machines and Logistic Regression completely dominate the board. While the Linear SVM took the #1 and #2 spots, Logistic Regression proved to be a highly competitive algorithm, breaking into the top 3. Text classification naturally produces highly dimensional, sparse datasets, and these results confirm that linear models are exceptionally well-suited for drawing geometric boundaries and calculating probabilities in this specific type of space.

**2. Advanced Features Outperform Baselines:** Our feature engineering efforts clearly paid off. The top half of the leaderboard is exclusively owned by our more advanced text representations: TF-IDF and N-grams. Conversely, our simplest baseline representations, One-Hot encoding and Bag-of-Words (BoW), are clustered entirely at the bottom. This proves that mathematically penalizing overly common words (TF-IDF) and capturing multi-word phrases (N-grams) provides a significantly richer learning signal than raw word counts or mere binary presence.

**3. The L1 Regularization Edge:** The absolute champion model is the **Linear SVM (L1) (TF-IDF)**. This highlights a crucial concept in NLP: because TF-IDF matrices contain thousands of sparse word columns, the L1 penalty acts as an aggressive, built-in feature selector. By pushing the weights of noisy, uninformative words to exactly zero, it prevents overfitting and gives the model a microscopic but decisive edge over the L2 penalty. Furthermore, the ROC-AUC scores across the board are exceptional (all > 0.989), indicating extreme confidence in their probability distributions.

Based on this leaderboard, we will select our absolute Top 3 configurations to advance to the exhaustive GridSearchCV phase to see if we can squeeze out even more performance:

* **Linear SVM (L1) (TF-IDF)**
* **Linear SVM (L2) (TF-IDF)**
* **Logistic Regression (L2) (grams)**

## Hyperparameter Tuning: Grid Search

In [ ]:
top_meta_models_config = {
    'Linear SVM (L1) - TF-IDF + Meta': {
        'model': LinearSVC(penalty='l1', dual=False, max_iter=5000, class_weight='balanced', random_state=42),
        'X_train': X_train_tfidf_final,
        'y_train': y_train_tfidf,
        'param_grid': {
            'C': [0.01, 0.1, 1.0, 10.0],
            'tol': [1e-4, 1e-3]
        }
    },

    'Linear SVM (L2) - TF-IDF + Meta': {
        'model': LinearSVC(penalty='l2', max_iter=5000, class_weight='balanced', random_state=42),
        'X_train': X_train_tfidf_final,
        'y_train': y_train_tfidf,
        'param_grid': {
            'C': [0.01, 0.1, 1.0, 10.0],
            'loss': ['hinge', 'squared_hinge'],
            'tol': [1e-4, 1e-3]
        }
    },

    'Logistic Regression (L2) - N-grams + Meta': {
        'model': LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42, solver='lbfgs'),
        'X_train': X_train_grams_final,
        'y_train': y_train_tfidf,
        'param_grid': {
            'C': [0.01, 0.1, 1.0, 10.0],
            'tol': [1e-4, 1e-3]
        }
    }
}

In [ ]:
best_meta_estimators = {}
meta_grid_results = []

print("Starting Grid Search for Meta-Feature Models... \n")

for name, config in top_meta_models_config.items():
    print(f"Tuning {name}...")

    grid = GridSearchCV(
        estimator=config['model'],
        param_grid=config['param_grid'],
        cv=5,
        scoring='f1_weighted',
        n_jobs=-1,
        verbose=1
    )

    # Fit the grid search using the specific X_train for this configuration
    grid.fit(config['X_train'], config['y_train'])

    # Store the best estimator
    best_meta_estimators[name] = grid.best_estimator_

    # Extract Mean and Std Deviation
    best_idx = grid.best_index_
    mean_score = grid.cv_results_['mean_test_score'][best_idx]
    std_score = grid.cv_results_['std_test_score'][best_idx]

    # Save the results
    meta_grid_results.append({
        'Model Configuration': name,
        'F1 Score (5-fold CV)': f"{mean_score:.4f} ﷿﷿ {std_score:.4f}",
        'Best Parameters': grid.best_params_
    })

    print(f"Done! Best Params: {grid.best_params_}\n")

In [ ]:
pd.set_option('display.max_colwidth', None)
meta_results_df = pd.DataFrame(meta_grid_results).sort_values(by='F1 Score (5-fold CV)', ascending=False)
display(meta_results_df)

In [ ]:
final_metrics_data = []
model_results = {}

print("--- FINAL TEST SET SCORES (Grid Search Winners) ---\n")

for model_name, tuned_model in best_meta_estimators.items():

    if "N-grams" in model_name:
        X_test_current = X_test_grams_final
        y_test_current = y_test_grams
    elif "TF-IDF" in model_name:
        X_test_current = X_test_tfidf_final
        y_test_current = y_test_tfidf
    else:
        print(f"Skipping unknown feature set for {model_name}")
        continue

    y_pred = tuned_model.predict(X_test_current)

    if hasattr(tuned_model, "predict_proba"):
        y_scores_prob = tuned_model.predict_proba(X_test_current)
    else:
        y_scores_raw = tuned_model.decision_function(X_test_current)
        y_scores_prob = softmax(y_scores_raw, axis=1)

    acc = accuracy_score(y_test_current, y_pred)
    prec = precision_score(y_test_current, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test_current, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test_current, y_pred, average='weighted', zero_division=0)
    roc_auc = roc_auc_score(y_test_current, y_scores_prob, multi_class='ovr', average='weighted')

    final_metrics_data.append({
        'Model': model_name,
        'Accuracy': acc,
        'Precision (Weighted)': prec,
        'Recall (Weighted)': rec,
        'F1 Score (Weighted)': f1,
        'ROC-AUC (Weighted)': roc_auc
    })

    model_results[model_name] = {
        'y_test': y_test_current,
        'y_pred': y_pred
    }

final_metrics_df = pd.DataFrame(final_metrics_data).set_index('Model')
sorted_metrics_df = final_metrics_df.sort_values(by='F1 Score (Weighted)', ascending=False)

display(sorted_metrics_df)

In [ ]:
best_model_name = sorted_metrics_df.index[0]

print(f"\n{'='*70}")
print(f"BEST MODEL: {best_model_name}")
print(f"{'='*70}\n")

best_y_test = model_results[best_model_name]['y_test']
best_y_pred = model_results[best_model_name]['y_pred']

In [ ]:
cm = confusion_matrix(best_y_test, best_y_pred)

plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=topics, yticklabels=topics)

plt.title(f'Confusion Matrix: {best_model_name}', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Actual Topic (True Label)', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Topic', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()



In [ ]:
report_dict = classification_report(best_y_test, best_y_pred, target_names=topics, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()

print("Classification Report:")
display(report_df.round(4))